In [5]:
import duckdb
import numpy as np
import pandas as pd

con = duckdb.connect("../olist.duckdb")

In [6]:
tables = {"orders": "olist_orders_dataset.csv",
          "customers": "olist_customers_dataset.csv",
          "geolocation": "olist_geolocation_dataset.csv",
          "order_items": "olist_order_items_dataset.csv",
          "payments": "olist_order_payments_dataset.csv",
          "reviews": "olist_order_reviews_dataset.csv",
          "products": "olist_products_dataset.csv",
          "sellers": "olist_sellers_dataset.csv",
          "category": "product_category_name_translation.csv"}

for table_name, file_name in tables.items():
    con.sql(f"CREATE OR REPLACE TABLE {table_name} AS SELECT * FROM read_csv_auto('../data/{file_name}')")
    
    n_rows = con.sql(f"SELECT COUNT(*) AS n FROM {table_name}").df()["n"][0]
    print(f"Table {table_name} has {n_rows} rows.")

Table orders has 99441 rows.
Table customers has 99441 rows.
Table geolocation has 1000163 rows.
Table order_items has 112650 rows.
Table payments has 103886 rows.
Table reviews has 99224 rows.
Table products has 32951 rows.
Table sellers has 3095 rows.
Table category has 71 rows.


In [8]:
con.sql("SHOW TABLES")

nameVARCHAR
category
customers
geolocation
order_items
orders
payments
products
reviews
sellers


In [17]:
con.sql("DESCRIBE reviews").df()

,column_name,column_type,null,key,default,extra
0,review_id,VARCHAR,YES,None,None,None
1,order_id,VARCHAR,YES,None,None,None
2,review_score,BIGINT,YES,None,None,None
3,review_comment_title,VARCHAR,YES,None,None,None
4,review_comment_message,VARCHAR,YES,None,None,None
5,review_creation_date,TIMESTAMP,YES,None,None,None
6,review_answer_timestamp,TIMESTAMP,YES,None,None,None


In [15]:
con.sql("""
    SELECT table_name, column_name
    FROM information_schema.columns
    WHERE table_schema = 'main'
    ORDER BY column_name
""").df()

,table_name,column_name
0,customers,customer_city
1,customers,customer_id
2,orders,customer_id
3,customers,customer_state
4,customers,customer_unique_id
5,customers,customer_zip_code_prefix
6,order_items,freight_value
7,geolocation,geolocation_city
8,geolocation,geolocation_lat
9,geolocation,geolocation_lng


In [23]:
con.close()

In [18]:
con.sql("""
    SELECT COUNT(*) AS rows, COUNT(DISTINCT order_id) AS unique_orders
    FROM reviews
""").df()

,rows,unique_orders
0,99224,98673


In [22]:
def check(table, column):
    return con.sql(f"""
        SELECT '{table}.{column}' AS col,
               COUNT(*) AS rows,
               COUNT(DISTINCT {column}) AS unique_values
        FROM {table}
    """).df()

# check("order_items", "seller_id")
check("orders", "customer_id")

,col,rows,unique_values
0,orders.customer_id,99441,99441
